In [ ]:
!pip install boto3

In [ ]:
import psycopg2
import os
import boto3
import io

In [ ]:
from google.colab import userdata

In [ ]:
DB_HOST = userdata.get('DB_HOST')
DB_USER = userdata.get('DB_USER')
DB_PASS = userdata.get('DB_PASS')
AWS_KEY = userdata.get('AWS_KEY')
AWS_SECRET_KEY = userdata.get('AWS_SECRET_KEY')

In [ ]:

### conectando e criando database
try:
    con = psycopg2.connect(
        host=DB_HOST,
        database='postgres',
        user=DB_USER,
        password=DB_PASS
    )

    con.autocommit = True

    cur = con.cursor()
    cur.execute('CREATE DATABASE inventario;')
    print("Banco de dados criado com sucesso!")

except psycopg2.OperationalError as e:
    print(f"Erro de conexão (verifique Security Groups e IP): {e}")
except Exception as e:
    print(f"Ocorreu um erro: {e}")
finally:
    if 'cur' in locals():
        cur.close()
    if 'con' in locals():
        con.close()
        print("Conexão encerrada.")

In [ ]:
### conectando e criando tabelas
try:
    con = psycopg2.connect(
        host=DB_HOST,
        database='inventario',
        user=DB_USER,
        password=DB_PASS
    )

    con.autocommit = True

    cur = con.cursor()
    cur.execute('CREATE TABLE ARQUIVOS(id_arquivo INT, nomearquivo VARCHAR(256));')

    print("Tabela criada com sucesso!")

except psycopg2.OperationalError as e:
    print(f"Erro de conexão (verifique Security Groups e IP): {e}")
except Exception as e:
    print(f"Ocorreu um erro: {e}")
finally:
    if 'cur' in locals():
        cur.close()
    if 'con' in locals():
        con.close()
        print("Conexão encerrada.")

In [ ]:
##CONECTANDO NO BUCKET COM AS IMAGENS

s3 = boto3.resource(
  service_name = 's3',
  region_name = 'us-east-2',
  aws_access_key_id = AWS_KEY,
  aws_secret_access_key = AWS_SECRET_KEY
  )

bucket = 'imagensengdados777'
prefix = 'imagens/'

try:
    con = psycopg2.connect(
        host=DB_HOST,
        database='inventario',
        user=DB_USER,
        password=DB_PASS
    )

    con.autocommit = True
    cur = con.cursor()

    # Get the maximum existing id from the database
    cur.execute("SELECT MAX(id_arquivo) FROM ARQUIVOS;")
    max_id = cur.fetchone()[0]
    if max_id is None:
        id = 0
    else:
        id = max_id

    cur.execute("SELECT nomearquivo FROM ARQUIVOS;")
    existing_filenames = {row[0] for row in cur.fetchall()}

    for object_s3 in s3.Bucket(bucket).objects.filter(Prefix=prefix):
        if object_s3.key.endswith('jpg') or object_s3.key.endswith('JPG'):
            filename = object_s3.key.split('/')[1] # tirando o diretorio e mostrando so o nome do arquivo

            if filename not in existing_filenames:
                id += 1
                cur.execute("INSERT INTO arquivos (id_arquivo, nomearquivo) VALUES (%s, %s)", (id, filename))
                print(f"Inserted: {filename} with ID: {id}")
            else:
                print(f"Já existe: {filename}")

except psycopg2.OperationalError as e:
    print(f"Erro de conexão (verifique Security Groups e IP): {e}")
except Exception as e:
    print(f"Ocorreu um erro: {e}")
finally:
    if 'cur' in locals():
        cur.close()
    if 'con' in locals():
        con.close()
        print("Conexão encerrada.")

In [ ]:
### VERIFICANDO
try:
    con = psycopg2.connect(
        host=DB_HOST,
        database='inventario',
        user=DB_USER,
        password=DB_PASS
    )

    con.autocommit = True

    cur = con.cursor()
    cur.execute('SELECT * FROM ARQUIVOS;')

    recset = cur.fetchall()

    for rec in recset:
        print(rec)

except psycopg2.OperationalError as e:
    print(f"Erro de conexão (verifique Security Groups e IP): {e}")
except Exception as e:
    print(f"Ocorreu um erro: {e}")
finally:
    if 'cur' in locals():
        cur.close()
    if 'con' in locals():
        con.close()
        print("Conexão encerrada.")

